In [1]:
print("PDF GROQ RAG")

PDF GROQ RAG


In [2]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()

True

In [6]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [7]:
llm=ChatGroq(model='llama-3.1-8b-instant',
             temperature=0.4)

In [8]:
llm.invoke("Who is meta founder").content

'Mark Zuckerberg is the co-founder, chairman and CEO of Meta, which he originally founded as Facebook in 2004.'

In [9]:
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
embed=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\HP\AppData\Local\Temp\ipykernel_2452\276418967.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
loader=PyMuPDFLoader('Hamza_Personal_Profile.pdf')

In [11]:
document=loader.load()

In [12]:
document[0].page_content

"Muhammad Hamza - Personal Profile\nHi, I'm Hamza\nData Scientist | Machine Learning Enthusiast | Developer\n- Education:\n  - BS Data Science - Emerson University, Multan\n  - GPA (1st Semester): 4.00/4.00\n  - Currently in 2nd year; will be in 3rd semester in September 2025.\n- Languages: English, Urdu, Punjabi\n- Passionate about Machine Learning, Deep Learning (CNNs), and Data Science\n- Goal: Become a great coder and innovator in tech.\n- Currently working on Arduino IoT projects, ML, DL\n------------------------------------------------------------\nMy Projects\nMachine Learning & AI Projects\n- Titanic Survival  Predict who survived using ML\n- Score Prediction  Estimate academic scores based on input features\n- Banana Quality Classification  Detect banana quality using image classification (CNN)\n- Match Maker  Predict compatibility between individuals\n- Weather Prediction  Predict temperature, humidity using regression\n- Student Attendance System (ID Card based)  Track stude

In [13]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=40)

In [14]:
chunks=text_splitter.split_documents(document)

In [15]:
for c in chunks:
    print(c)
    print('---')

page_content='Muhammad Hamza - Personal Profile
Hi, I'm Hamza
Data Scientist | Machine Learning Enthusiast | Developer
- Education:
  - BS Data Science - Emerson University, Multan
  - GPA (1st Semester): 4.00/4.00
  - Currently in 2nd year; will be in 3rd semester in September 2025.
- Languages: English, Urdu, Punjabi
- Passionate about Machine Learning, Deep Learning (CNNs), and Data Science' metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': '', 'creationdate': 'D:20250812091113', 'source': 'Hamza_Personal_Profile.pdf', 'file_path': 'Hamza_Personal_Profile.pdf', 'total_pages': 3, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20250812091113', 'page': 0}
---
page_content='- Goal: Become a great coder and innovator in tech.
- Currently working on Arduino IoT projects, ML, DL
------------------------------------------------------------
My Projects
Machine Learning

In [16]:
vectorstores=Chroma.from_documents(chunks,embed)

In [17]:
retriever=vectorstores.as_retriever(search_type='similarity',search_kwargs={'k':3})

In [18]:
from langchain.chains import RetrievalQA
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
memory=ConversationBufferMemory()

C:\Users\HP\AppData\Local\Temp\ipykernel_2452\1724443673.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory()


In [33]:
template="""You are a PDF BOT. You will get a context and always reply to user query based on it.
Reply Should not be more than 50 words!
Here is query: {question}
Here is context: {context}"""
prompt=PromptTemplate(template=template,input_variables=['question','context'])

In [34]:
pdf_bot=RetrievalQA.from_chain_type(llm=llm,retriever=retriever,memory=memory,chain_type_kwargs={'prompt':prompt})

In [35]:
query='Tell me something about hamza skills'


In [36]:
similar_doc=vectorstores.similarity_search(query,k=3)
for s in similar_doc:
    print(s)
    print('-'*10)

page_content='Muhammad Hamza - Personal Profile
Hi, I'm Hamza
Data Scientist | Machine Learning Enthusiast | Developer
- Education:
  - BS Data Science - Emerson University, Multan
  - GPA (1st Semester): 4.00/4.00
  - Currently in 2nd year; will be in 3rd semester in September 2025.
- Languages: English, Urdu, Punjabi
- Passionate about Machine Learning, Deep Learning (CNNs), and Data Science' metadata={'subject': '', 'page': 0, 'file_path': 'Hamza_Personal_Profile.pdf', 'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'source': 'Hamza_Personal_Profile.pdf', 'keywords': '', 'format': 'PDF 1.3', 'trapped': '', 'moddate': '', 'creator': '', 'modDate': '', 'total_pages': 3, 'creationDate': 'D:20250812091113', 'creationdate': 'D:20250812091113', 'title': '', 'author': ''}
----------
page_content='Web Development: FastAPI, Flask, Vercel
Version Control: Git, GitHub
Development Tools: Jupyter Notebook, Google Colab, VS Code
IoT: C, C++, Microcontrollers, ESP8266, ESP32, Arduino Uno

In [37]:
response=pdf_bot.invoke(query)

In [38]:
print('--------PDF BOT---------')
print(response['result'])

--------PDF BOT---------
Hamza skills include: 
- Data Science and Machine Learning
- Web Development (FastAPI, Flask, Vercel)
- Version Control (Git, GitHub)
- Development Tools (Jupyter Notebook, Google Colab, VS Code)
- IoT (C, C++, Microcontrollers, ESP8266, ESP32, Arduino Uno)
